In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:
DATA_PATH = "/content/drive/MyDrive/DINO_Features"

train = np.load(os.path.join(DATA_PATH, "train_data.npz"))
valid = np.load(os.path.join(DATA_PATH, "valid_data.npz"))
test = np.load(os.path.join(DATA_PATH, "test_data.npz"))

In [ ]:
train_features = train["features"]
train_labels = train["labels"]

valid_features = valid["features"]
valid_labels = valid["labels"]

test_features = test["features"]
test_labels = test["labels"]

In [ ]:
print(train_features.shape)
print(train_labels.shape)

print(valid_features.shape)
print(valid_labels.shape)

print(test_features.shape)
print(test_labels.shape)

(4368, 768)
(4368,)
(182, 768)
(182,)
(215, 768)
(215,)


In [ ]:
train_features = torch.tensor(train_features, dtype=torch.float32)
train_labels = torch.tensor(train_labels, dtype=torch.long)

valid_features = torch.tensor(valid_features, dtype=torch.float32)
valid_labels = torch.tensor(valid_labels, dtype=torch.long)

test_features = torch.tensor(test_features, dtype=torch.float32)
test_labels = torch.tensor(test_labels, dtype=torch.long)

In [ ]:
train_dataset = TensorDataset(
    train_features,
    train_labels
)

valid_dataset = TensorDataset(
    valid_features,
    valid_labels
)

test_dataset = TensorDataset(
    test_features,
    test_labels
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
class CoffeeMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(768,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(256,7)

        )

    def forward(self,x):

        return self.model(x)

In [ ]:
model = CoffeeMLP().to(device)

print(model)

CoffeeMLP(
  (model): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=256, out_features=7, bias=True)
  )
)


In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 30

best_accuracy = 0

In [ ]:
for epoch in range(epochs):

    model.train()

    train_loss = 0

    for features,labels in train_loader:

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    #############################################
    # Validation
    #############################################

    model.eval()

    predictions = []
    actual = []

    with torch.no_grad():

        for features,labels in valid_loader:

            features = features.to(device)

            outputs = model(features)

            pred = torch.argmax(outputs,dim=1)

            predictions.extend(pred.cpu().numpy())

            actual.extend(labels.numpy())

    accuracy = accuracy_score(actual,predictions)

    print(

        f"Epoch {epoch+1}/{epochs}"

        f" | Loss: {train_loss:.4f}"

        f" | Validation Accuracy: {accuracy:.4f}"

    )

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        torch.save(

            model.state_dict(),

            "/content/drive/MyDrive/DINO_Features/mlp_classifier.pth"

        )

print("Training Finished")

Epoch 1/30 | Loss: 33.6821 | Validation Accuracy: 0.9396
Epoch 2/30 | Loss: 9.7672 | Validation Accuracy: 0.9286
Epoch 3/30 | Loss: 5.7923 | Validation Accuracy: 0.9286
Epoch 4/30 | Loss: 5.3562 | Validation Accuracy: 0.9341
Epoch 5/30 | Loss: 5.2815 | Validation Accuracy: 0.9341
Epoch 6/30 | Loss: 2.7271 | Validation Accuracy: 0.9286
Epoch 7/30 | Loss: 2.0135 | Validation Accuracy: 0.9396
Epoch 8/30 | Loss: 1.7261 | Validation Accuracy: 0.9341
Epoch 9/30 | Loss: 1.6104 | Validation Accuracy: 0.9121
Epoch 10/30 | Loss: 2.1712 | Validation Accuracy: 0.9396
Epoch 11/30 | Loss: 2.4358 | Validation Accuracy: 0.9011
Epoch 12/30 | Loss: 1.8294 | Validation Accuracy: 0.9341
Epoch 13/30 | Loss: 2.2132 | Validation Accuracy: 0.9451
Epoch 14/30 | Loss: 1.4834 | Validation Accuracy: 0.9396
Epoch 15/30 | Loss: 2.0430 | Validation Accuracy: 0.9451
Epoch 16/30 | Loss: 1.1045 | Validation Accuracy: 0.9341
Epoch 17/30 | Loss: 1.1194 | Validation Accuracy: 0.9231
Epoch 18/30 | Loss: 0.9746 | Validation

In [ ]:
model.load_state_dict(

    torch.load(

        "/content/drive/MyDrive/DINO_Features/mlp_classifier.pth"

    )

)

<All keys matched successfully>

In [ ]:
model.eval()

predictions = []
actual = []

with torch.no_grad():

    for features,labels in test_loader:

        features = features.to(device)

        outputs = model(features)

        pred = torch.argmax(outputs,dim=1)

        predictions.extend(pred.cpu().numpy())

        actual.extend(labels.numpy())

accuracy = accuracy_score(actual,predictions)

print("Test Accuracy :",accuracy)

Test Accuracy : 0.8976744186046511


In [ ]:
class_names = [
    "cut",
    "good",
    "husk",
    "immature",
    "parchment",
    "partial-black",
    "shell"
]

print(

    classification_report(

        actual,

        predictions,

        target_names=class_names

    )

)

               precision    recall  f1-score   support

          cut       0.85      0.96      0.90        23
         good       0.94      1.00      0.97        58
         husk       0.98      0.80      0.88        54
     immature       0.90      0.82      0.86        11
    parchment       0.92      1.00      0.96        11
partial-black       0.83      0.81      0.82        31
        shell       0.81      0.93      0.86        27

     accuracy                           0.90       215
    macro avg       0.89      0.90      0.89       215
 weighted avg       0.90      0.90      0.90       215

